In [1]:
import gc
import os
import sys

import numpy as np
import optuna
import pandas as pd
import torch
from sklearn.model_selection import KFold

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
%load_ext autoreload
%autoreload 2
from drGAT import No_atten_drGAT
from drGAT.load_data import load_data
from drGAT.sampler import RandomSampler

'cuda'

In [3]:
drugAct, pos_num, null_mask, S_d, S_c, S_g, A_cg, A_dg = load_data("ctrp")

In [ ]:
PATH = "../ctrp_data/"

In [14]:
def objective(trial):
    params = {
        "n_drug": S_d.shape[0],
        "n_cell": S_c.shape[0],
        "n_gene": S_g.shape[0],
        "dropout1": trial.suggest_categorical("dropout1", [0.1, 0.2, 0.3, 0.4, 0.5]),
        "dropout2": trial.suggest_categorical("dropout2", [0.1, 0.2, 0.3, 0.4, 0.5]),
        "hidden1": trial.suggest_categorical(
            "hidden1",
            [256, 512, 1028],
        ),
        "hidden2": trial.suggest_categorical(
            "hidden2",
            [
                128,
                256,
                512,
            ],
        ),
        "hidden3": trial.suggest_categorical(
            "hidden3",
            [
                64,
                128,
                256,
            ],
        ),
        "epochs": trial.suggest_categorical("epochs", [10, 50, 100, 200, 500]),
        "heads": trial.suggest_categorical("heads", [1, 2, 3, 4, 5]),
        "activation": trial.suggest_categorical(
            "activation", ["relu", "gelu", "swish"]
        ),
        "optimizer": trial.suggest_categorical("optimizer", ["Adam", "AdamW", "SGD"]),
        "lr": trial.suggest_float("lr", 1e-5, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "scheduler": trial.suggest_categorical(
            "scheduler", [None, "Cosine", "Step", "Plateau"]
        ),
        "gnn_layer": trial.suggest_categorical(
            "gnn_layer",
            ["GCN", "MPNN"],
        ),
    }

    # スケジューラ関連パラメータの条件付き追加
    if params["scheduler"] == "Cosine":
        params["T_max"] = trial.suggest_int("T_max", 20, 50)
    elif params["scheduler"] == "Step":
        params["scheduler_gamma"] = trial.suggest_float("gamma_step", 0.1, 0.95)
        params["step_size"] = trial.suggest_int("step_size", 10, 30)
    elif params["scheduler"] == "Plateau":
        params["patience"] = trial.suggest_int("patience_plateau", 3, 10)
        params["threshold"] = trial.suggest_float(
            "thresh_plateau", 1e-4, 1e-2, log=True
        )

    if params["hidden1"] < params["hidden2"] or params["hidden2"] < params["hidden3"]:
        raise optuna.TrialPruned("Invalid layer size configuration")

    if params["optimizer"] in ["Adam", "AdamW"]:
        params["amsgrad"] = trial.suggest_categorical("amsgrad", [True, False])

    if params["optimizer"] == "SGD":
        params["momentum"] = trial.suggest_float("momentum", 0.8, 0.95)
        params["nesterov"] = trial.suggest_categorical("nesterov", [True, False])

    # 隠れ層サイズとバッチサイズの関係を制約
    if (params["hidden1"] > 512) and (params["hidden2"] > 256):
        raise optuna.TrialPruned("Memory intensive configuration")

    try:
        k = 5
        kfold = KFold(n_splits=k, shuffle=True, random_state=42)

        res = pd.DataFrame()
        for train_index, test_index in kfold.split(np.arange(pos_num)):
            sampler = RandomSampler(
                drugAct,
                train_index,
                test_index,
                null_mask,
                S_d,
                S_c,
                S_g,
                A_cg,
                A_dg,
                PATH,
            )
            _, best_metrics, _ = No_atten_drGAT.train(
                sampler, params=params, device=device, verbose=False
            )

            res = pd.concat(
                [
                    res,
                    pd.DataFrame(best_metrics, index=["acc", "f1", "auroc", "aupr"]).T,
                ]
            )

        return [float(i) for i in res.mean()]

    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            print(f"Pruned trial {trial.number}: CUDA OOM")

            # メモリ解放処理
            with torch.cuda.device("cuda"):
                torch.cuda.empty_cache()
            gc.collect()

            # Pruning通知
            raise optuna.TrialPruned(f"OOM at trial {trial.number}")

        else:
            raise e

In [15]:
name = "CTRP"
study = optuna.create_study(
    directions=["maximize"] * 4,
    sampler=optuna.samplers.TPESampler(),
    pruner=optuna.pruners.HyperbandPruner(),
    storage="sqlite:///{}_{}.sqlite3".format(name, "GCN_MPNN"),
    study_name=name,
    load_if_exists=True,
)
study.optimize(objective, n_trials=100, n_jobs=8)

[I 2025-03-21 17:11:17,244] Using an existing study with name 'CTRP' instead of creating a new one.
/data/inouey2/conda/envs/genex/lib/python3.10/site-packages/optuna/distributions.py:700: UserWarning: The distribution is specified by [10, 200] and step=50, but the range is not divisible by `step`. It will be replaced by [10, 160].
  warnings.warn(


Using:  cuda
Using:  cuda
Using:  cuda
Using:  cuda
Using:  cuda
Using:  cuda
Using:  cuda
Using:  cuda







  0%|          | 0/160 [00:00<?, ?it/s]



  0%|          | 0/160 [00:00<?, ?it/s]

  0%|          | 0/160 [00:00<?, ?it/s]


  0%|          | 0/160 [00:00<?, ?it/s]






  0%|          | 0/160 [00:00<?, ?it/s]





  0%|          | 0/160 [00:00<?, ?it/s]

  1%|          | 1/160 [00:02<05:20,  2.02s/it]






  1%|          | 1/160 [00:02<05:51,  2.21s/it]


  1%|          | 1/160 [00:02<06:04,  2.29s/it]



  1%|          | 1/160 [00:02<06:13,  2.35s/it]




  1%|          | 1/160 [00:02<06:19,  2.39s/it]





  1%|          | 1/160 [00:02<06:22,  2.41s/it]

  1%|▏         | 2/160 [00:03<03:54,  1.48s/it]


  1%|▏         | 2/160 [00:03<04:01,  1.53s/it]



  1%|▏         | 2/160 [00:03<04:09,  1.58s/it]






  1%|▏         | 2/160 [00:03<04:11,  1.59s/it]




  1%|▏         | 2/160 [00:03<04:18,  1.64s/it]





  1%|▏         | 2/160 [00:03<04:13,  1.61s/it]

  2%|▏         | 3/160 [00:04<03:31,  1.35s/it]


  2%|▏         | 3/160 [00:04<03:35,  1.37s/it]





  2%|▏         |

Best model found at epoch 160
Best model found at epoch 160




 96%|█████████▌| 153/160 [02:57<00:09,  1.33s/it]


 98%|█████████▊| 156/160 [02:57<00:05,  1.29s/it]




 95%|█████████▌| 152/160 [02:57<00:10,  1.28s/it][I 2025-03-21 17:14:18,771] Trial 42 finished with values: [0.8270667630754399, 0.900659490879813, 0.90291864257888, 0.8251705653021443] and parameters: {'dropout1': 0.2, 'dropout2': 0.4, 'hidden1': 1028, 'hidden2': 128, 'hidden3': 128, 'epochs': 160, 'activation': 'relu', 'optimizer': 'Adam', 'lr': 0.007690591084791501, 'weight_decay': 0.0002050508724043216, 'scheduler': 'Step', 'gnn_layer': 'GCN', 'gamma_step': 0.10467372714993511, 'step_size': 30, 'amsgrad': True, 'early_stop_threshold': 0.44578231983891903}. 
[I 2025-03-21 17:14:18,836] Trial 46 finished with values: [0.8271270185586889, 0.8999993805793676, 0.9026184761639531, 0.8252208346024977] and parameters: {'dropout1': 0.2, 'dropout2': 0.4, 'hidden1': 1028, 'hidden2': 128, 'hidden3': 128, 'epochs': 160, 'activation': 'relu', 'optimizer': 'Adam', 'lr': 0.009751245445096032

...

[I 2025-03-21 17:18:58,505] Trial 213 finished with values: [-inf, -inf, -inf, -inf] and parameters: {'dropout1': 0.3, 'dropout2': 0.4, 'hidden1': 256, 'hidden2': 256, 'hidden3': 256, 'epochs': 110, 'activation': 'gelu', 'optimizer': 'Adam', 'lr': 2.7367841168516213e-05, 'weight_decay': 0.0001279741124963853, 'scheduler': 'Cosine', 'gnn_layer': 'MPNN', 'T_max': 25, 'amsgrad': True}. 


  1%|          | 1/160 [00:01<04:02,  1.52s/it]



  5%|▌         | 6/110 [00:09<02:36,  1.51s/it]A



  1%|          | 1/160 [00:01<04:14,  1.60s/it][W 2025-03-21 17:18:59,163] Trial 220 failed with parameters: {'dropout1': 0.3, 'dropout2': 0.1, 'hidden1': 256, 'hidden2': 256, 'hidden3': 256, 'epochs': 110, 'activation': 'gelu', 'optimizer': 'Adam', 'lr': 0.007409430035542206, 'weight_decay': 0.00013334491055421344, 'scheduler': 'Cosine', 'gnn_layer': 'MPNN', 'T_max': 32, 'amsgrad': False} because of the following error: ValueError('Input contains NaN.').
Traceback (most recent call last):
  File "/data

CUDA out of memory
CUDA out of memory
CUDA out of memory
CUDA out of memory


CUDA out of memory


[I 2025-03-21 17:19:02,379] Trial 225 finished with values: [-inf, -inf, -inf, -inf] and parameters: {'dropout1': 0.4, 'dropout2': 0.1, 'hidden1': 256, 'hidden2': 256, 'hidden3': 64, 'epochs': 160, 'activation': 'relu', 'optimizer': 'Adam', 'lr': 0.006826093147308835, 'weight_decay': 0.0001885309218843481, 'scheduler': 'Cosine', 'gnn_layer': 'MPNN', 'T_max': 29, 'amsgrad': False}. 
[I 2025-03-21 17:19:02,397] Trial 223 finished with values: [-inf, -inf, -inf, -inf] and parameters: {'dropout1': 0.4, 'dropout2': 0.1, 'hidden1': 256, 'hidden2': 256, 'hidden3': 64, 'epochs': 160, 'activation': 'relu', 'optimizer': 'Adam', 'lr': 0.0007938875220908824, 'weight_decay': 0.0008387472781195097, 'scheduler': 'Cosine', 'gnn_layer': 'MPNN', 'T_max': 29, 'amsgrad': False}. 
[I 2025-03-21 17:19:02,412] Trial 222 finished with values: [-inf, -inf, -inf, -inf] and parameters: {'dropout1': 0.4, 'dropout2': 0.1, 'hidden1': 256, 'hidden2': 256, 'hidden3': 64, 'epochs': 160, 'activation': 'relu', 'optimize

ValueError: Input contains NaN.

## Eval

In [ ]:
# test_drug = test_data.values[:, 0]
# test_cell = test_data.values[:, 1]

# test_labels = np.load("data/test_labels.npy")
# test_labels = torch.tensor(test_labels).float()
# test = [drug, cell, gene, edge_index, test_drug, test_cell, test_labels]

In [ ]:
# prob, res, test_attention = drGAT.eval(model, test)